# Team 1 - Capstone Project: Academic Q&A Assistant (UABC)
## Modules 15, 16, 17, 18 - Complete & Polish (v9)

**Proyecto:** Asistente RAG institucional sobre el uso de IA en la UABC
**Base de conocimiento:** 3 documentos PDF institucionales
**Stack:** LangChain + OpenAI + ChromaDB + Streamlit

**Team 1 Members:**
- Garcia Canseco Eloisa del Carmen
- Inzunza Gonzalez Everardo
- Navarro Cota Christian Xavier
- Rivera Aguirre Flavio Abel

---

### Module 17 Objectives
- Debug and fix common RAG system issues based on test results
- Improve user experience through clear instructions and error handling
- Document the project in a reproducible and professional manner
- Reflect critically on LLM behavior and system performance

### v9 Feedback Submit Button (all v8 features preserved)
- Added "Send Comment" button in Rate Last Response section
- Added "Respuesta enviada" confirmation after comment submission
- UABC logo, team members, dynamic settings (from v8/v7)

---
## 1. Environment Setup

In [1]:
# Instalar dependencias (filtrar warnings internos de Colab que NO afectan al proyecto)
!pip install -U -q openai langchain langchain-openai langchain-community chromadb pypdf tiktoken python-dotenv 2>&1 | grep -v "dependency conflicts" | grep -v "which is incompatible" | grep -v "requires " | grep -v "google-colab" | grep -v "opentelemetry" | grep -v "ERROR: pip"
print("\n✅ Dependencias instaladas correctamente")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6

In [2]:
import requests, openai, langchain, chromadb, pypdf
print("requests:", requests.__version__)
print("openai:", openai.__version__)
print("langchain:", langchain.__version__)
print("chromadb:", chromadb.__version__)
print("pypdf:", pypdf.__version__)

requests: 2.32.5
openai: 2.21.0
langchain: 1.2.10
chromadb: 1.5.0
pypdf: 6.7.1


In [3]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OpenAI API Key: ")

Introduce tu OpenAI API Key: ··········


In [4]:
print("OK" if os.getenv("OPENAI_API_KEY") else "FALTA API KEY")

OK


---
## 2. Load PDF Documents

**Instrucciones:** Antes de ejecutar la siguiente celda, sube los 3 archivos PDF a `/content/`:
1. `Boletin1_IA_2024-01-19.pdf`
2. `IA_Practica_Docente_2024-01-24.pdf`
3. `Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf`

> Puedes arrastrarlos al panel de archivos de Colab o usar el boton de subir.

In [5]:
import os

# Verificar que los PDFs estan en /content/
expected_files = [
    "Boletin1_IA_2024-01-19.pdf",
    "IA_Practica_Docente_2024-01-24.pdf",
    "Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf",
]

missing = [f for f in expected_files if not os.path.exists(f"/content/{f}")]
if missing:
    print("FALTAN archivos:", missing)
    print("Por favor sube los PDFs a /content/ antes de continuar.")
else:
    print("Todos los PDFs encontrados en /content/")
    for f in expected_files:
        size_kb = os.path.getsize(f"/content/{f}") / 1024
        print(f"  {f} ({size_kb:.0f} KB)")

Todos los PDFs encontrados en /content/
  Boletin1_IA_2024-01-19.pdf (831 KB)
  IA_Practica_Docente_2024-01-24.pdf (1160 KB)
  Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf (1068 KB)


---
## 3. Build RAG Pipeline

Arquitectura del pipeline:
```
PDFs --> PyPDFLoader --> RecursiveCharacterTextSplitter --> OpenAI Embeddings --> ChromaDB
                                                                                    |
User Question --> Retriever (k=4) --> format_context() --> Prompt Template --> GPT-4o-mini --> Answer
```

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- Document Loading ---
pdf_paths = [
    "/content/Boletin1_IA_2024-01-19.pdf",
    "/content/IA_Practica_Docente_2024-01-24.pdf",
    "/content/Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf",
]

docs = []
for p in pdf_paths:
    try:
        loader = PyPDFLoader(p)
        loaded = loader.load()
        docs.extend(loaded)
        print(f"  Loaded: {os.path.basename(p)} ({len(loaded)} pages)")
    except Exception as e:
        print(f"  ERROR loading {p}: {e}")

print(f"\nTotal pages loaded: {len(docs)}")

# --- Chunking ---
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

# --- Embeddings & Vector Store ---
emb = OpenAIEmbeddings(model="text-embedding-3-small")
vectordb = Chroma.from_documents(chunks, emb, persist_directory="./chroma_uabc_ai")
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print("Vector store created successfully")

# --- LLM ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
print("LLM ready: gpt-4o-mini")

  Loaded: Boletin1_IA_2024-01-19.pdf (14 pages)
  Loaded: IA_Practica_Docente_2024-01-24.pdf (15 pages)
  Loaded: Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf (12 pages)

Total pages loaded: 41
Total chunks: 97
Vector store created successfully
LLM ready: gpt-4o-mini


---
## 4. Prompt Template & Query Functions

**Module 17 Fix:** Se mejoro el prompt para reforzar que SOLO responda con base en las fuentes
y que incluya un fallback claro cuando no encuentra informacion.

In [7]:
# --- Prompt Template (mejorado en Module 17) ---
prompt = ChatPromptTemplate.from_template(
"""Eres un asistente institucional de la UABC sobre el uso de Inteligencia Artificial.
Responde SOLO con base en las fuentes proporcionadas.
Si la pregunta NO se puede responder con las fuentes, di exactamente:
"No encuentro informacion sobre ese tema en los documentos proporcionados."

Pregunta: {question}

Fuentes (extractos):
{context}

Instrucciones de formato:
- Da la respuesta en 3-8 lineas.
- Al final agrega una seccion "Referencias:" con el formato:
  (nombre_del_archivo.pdf, p.X; nombre_del_archivo.pdf, p.Y).
"""
)

def format_context(docs):
    """Format retrieved documents into a readable context string."""
    lines = []
    for d in docs:
        src = os.path.basename(d.metadata.get("source", ""))
        page = d.metadata.get("page", None)
        page_str = f"p.{page+1}" if isinstance(page, int) else "p.?"
        text = d.page_content.strip().replace("\n", " ")
        lines.append(f"- ({src}, {page_str}) {text}")
    return "\n".join(lines)

def ask(question: str) -> str:
    """Ask a question and return the answer."""
    retrieved = retriever.invoke(question)
    context = format_context(retrieved)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question, "context": context})
    return answer

def ask_with_sources(question: str):
    """Ask a question and return (answer, retrieved_docs)."""
    retrieved = retriever.invoke(question)
    context = format_context(retrieved)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question, "context": context})
    return answer, retrieved

print("Query functions ready.")

Query functions ready.


---
## 5. Quick Test

In [8]:
# Test rapido del pipeline
ans, srcs = ask_with_sources("Que riesgos o precauciones se mencionan sobre el uso de IA en investigacion?")
print(ans)

print("\n--- Fuentes recuperadas ---")
seen = set()
for d in srcs:
    src = os.path.basename(d.metadata.get("source", ""))
    page = d.metadata.get("page", "?")
    key = (src, page)
    if key not in seen:
        seen.add(key)
        print(f"  {src}, page: {page}")

Los riesgos y precauciones sobre el uso de IA en la investigación incluyen la necesidad de ser conscientes de los sesgos en los datos y en los sistemas de IA, así como la importancia de utilizar datos representativos y justos. Además, se destaca la relevancia de los comités de ética en la investigación para mitigar riesgos éticos relacionados con la recolección y tratamiento de datos, así como la protección de la información sensible de los participantes. También se menciona el riesgo de una brecha entre investigadores si no se incorpora el contexto de estudio y la supervisión humana adecuada.

Referencias: 
(Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf, p.8; Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf, p.10; Boletin1_IA_2024-01-19.pdf, p.10).

--- Fuentes recuperadas ---
  Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf, page: 7
  Incorporacion_IA_Procesos_de_Investigacion_2024-06-12-6.pdf, page: 9
  Boletin1_IA_2024-01-19.pdf, page: 9


---
## 6. Evaluation & Metrics (Module 16 Baseline)

Se evalua el pipeline con 10 preguntas representativas que cubren:
- Orientaciones institucionales
- Practica docente
- Investigacion y etica

In [9]:
import time
import pandas as pd
import os
import re

metrics_log = []

def has_any_citation(text: str) -> bool:
    """Check if the answer contains file+page citations."""
    if "Referencias:" in text:
        return True
    return re.search(r"\.pdf,\s*p\.\d+", text) is not None

def ask_eval(question: str, feedback: str = ""):
    """Ask a question, measure metrics, and log results."""
    t0 = time.time()

    retrieved = retriever.invoke(question)
    context = format_context(retrieved)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question, "context": context})

    latency = time.time() - t0
    has_citations = has_any_citation(answer)

    seen = set()
    sources = []
    docs_used = set()
    for d in retrieved:
        src = os.path.basename(d.metadata.get("source", ""))
        page = d.metadata.get("page", None)
        key = (src, page)
        if key in seen:
            continue
        seen.add(key)
        docs_used.add(src)
        if isinstance(page, int):
            sources.append(f"{src} (p.{page+1})")
        else:
            sources.append(src)

    metrics_log.append({
        "question": question,
        "latency_sec": round(latency, 3),
        "chunks_retrieved": len(retrieved),
        "unique_sources": len(sources),
        "unique_docs": len(docs_used),
        "has_citations": has_citations,
        "sources": "; ".join(sources),
        "answer": answer,
        "user_feedback": feedback,
    })

    return answer

# --- 10 Test Questions ---
questions = [
    "Que orientaciones iniciales se proponen para el uso academico de la IA en la universidad?",
    "Que recomendaciones se dan para integrar la IA generativa en la practica docente?",
    "Que riesgos eticos se mencionan al usar IA con datos de estudiantes o participantes?",
    "Que papel juegan los comites de etica en proyectos de investigacion que usan IA?",
    "Que se recomienda respecto a sesgos (bias) en datos y resultados generados por IA?",
    "Que buenas practicas se sugieren para el uso de IA en evaluacion o tareas academicas?",
    "Que etapas del proceso de investigacion pueden apoyarse con IA y con que precauciones?",
    "Que se recomienda sobre transparencia o trazabilidad (indicar cuando se uso IA)?",
    "Que limitaciones se advierten sobre confiar totalmente en respuestas generadas por IA?",
    "Segun los documentos, que acciones puede tomar una institucion para adoptar IA de forma responsable?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(ask_eval(q))

df_eval = pd.DataFrame(metrics_log)
df_eval.to_csv("evaluation_results_v9.csv", index=False)

print("\n" + "="*50)
print("EVALUATION SUMMARY")
print("="*50)
print(f"Questions evaluated:    {len(df_eval)}")
print(f"Avg latency (sec):      {df_eval['latency_sec'].mean():.2f}")
print(f"% with citations:       {100 * df_eval['has_citations'].mean():.0f}%")
print(f"Avg unique sources:     {df_eval['unique_sources'].mean():.1f}")
print(f"Avg unique docs:        {df_eval['unique_docs'].mean():.1f}")


Q: Que orientaciones iniciales se proponen para el uso academico de la IA en la universidad?
Las orientaciones iniciales para el uso académico de la Inteligencia Artificial (IA) en la UABC enfatizan la necesidad de adoptar una postura ética y responsable. Se reconoce el potencial de la IA para personalizar el aprendizaje y optimizar recursos, pero también se subraya la importancia de abordar los desafíos que surgen de su implementación. La universidad se compromete a garantizar que el uso de la IA sea equitativo y transparente, priorizando el bienestar de su comunidad y manteniendo sus valores fundamentales de excelencia académica y ética. 

Referencias: 
(Boletin1_IA_2024-01-19.pdf, p.1; Boletin1_IA_2024-01-19.pdf, p.4; Boletin1_IA_2024-01-19.pdf, p.12; Boletin1_IA_2024-01-19.pdf, p.9).

Q: Que recomendaciones se dan para integrar la IA generativa en la practica docente?
Para integrar la inteligencia artificial generativa en la práctica docente, se recomienda que los docentes se fami

### Evaluation Summary (Module 16 Baseline)

The RAG-based academic assistant was evaluated using 10 representative questions covering
institutional guidelines, teaching practice, and research topics. Key results:

| Metric | Value |
|--------|-------|
| Questions evaluated | 10 |
| Avg latency | ~4 sec |
| Citations present | 100% |
| Avg unique sources per query | ~3.5 |

These results confirm the RAG pipeline functions correctly and provides a quantitative
baseline for the Module 17 improvements below.

---
---
# MODULE 17: Complete & Polish

---
## Step 1: Fix Issues from Module 16 Testing

Based on evaluation results, we identified and addressed the following issues:

| Issue Found | Fix Applied |
|------------|-------------|
| Prompt occasionally produced answers without clear fallback for off-topic questions | Added explicit fallback instruction in prompt template |
| `chunk_overlap` was 80 (too low for coherent context) | Increased to 150 for better context continuity |
| No error handling for PDF loading failures | Added try/except per PDF in loading loop |
| Hardcoded paths only worked in Colab | Kept `/content/` paths but added file validation check |
| `streamlit_app.py` contained unused MockLLM class | Removed in v2 (clean code) |
| Streamlit did not show retrieved sources separately | Added `st.expander` to display sources in v2 |

### Test: Off-topic question handling

Let's verify the improved prompt rejects off-topic questions correctly:

In [10]:
# Test off-topic question handling (Module 17 fix)
off_topic_questions = [
    "Cual es la capital de Francia?",
    "Como se prepara una pizza?",
    "Quien gano el mundial de futbol en 2022?",
]

print("=== Off-topic Question Test ===\n")
for q in off_topic_questions:
    ans = ask(q)
    is_rejected = "no encuentro" in ans.lower() or "no lo encuentro" in ans.lower() or "no tengo" in ans.lower()
    status = "PASS (rejected)" if is_rejected else "FAIL (answered off-topic)"
    print(f"Q: {q}")
    print(f"A: {ans[:120]}...")
    print(f"Status: {status}\n")

=== Off-topic Question Test ===

Q: Cual es la capital de Francia?
A: No encuentro informacion sobre ese tema en los documentos proporcionados.

Referencias: (IA_Practica_Docente_2024-01-24....
Status: PASS (rejected)

Q: Como se prepara una pizza?
A: No encuentro informacion sobre ese tema en los documentos proporcionados. 

Referencias: (IA_Practica_Docente_2024-01-24...
Status: PASS (rejected)

Q: Quien gano el mundial de futbol en 2022?
A: No encuentro informacion sobre ese tema en los documentos proporcionados.

Referencias: (IA_Practica_Docente_2024-01-24....
Status: PASS (rejected)



---
## Step 2: Polish User Experience (Streamlit App v3)

Improvements made in `streamlit_app_v3.py` (building on v2):

**v2 fixes (carried forward):**
1. Clear instructions for users with example questions in sidebar
2. Loading indicators with `st.spinner`
3. Source display with `st.expander`
4. Error handling with `try/except` blocks
5. Removed dead code (MockLLM)
6. Fixed missing `plotly.graph_objects` import

**v3 new improvements:**
1. **Professional 3-column layout** - Sidebar / Chat Center / Metrics Right
2. **Gradient header banner** with institutional branding
3. **Scrollable chat history** (480px container, no flickering)
4. **Real-time metrics panel** on the right (queries, latency, satisfaction, errors)
5. **SLA badge** - color-coded green/yellow indicator
6. **Confidence bar** - animated with color coding (green/yellow/red)
7. **Source chips** - compact display of retrieved document chunks
8. **Inline feedback** - thumbs up/down buttons directly below chat
9. **Latency mini-chart** per query on the right panel
10. **Custom CSS** for polished, non-overlapping, professional UI

---
## Step 3: Launch Streamlit App (v9)

The Streamlit app (`streamlit_app_v9.py`) adds feedback UX improvements over v8:
- **Send Comment Button:** Explicit 📩 button replaces auto-submit behavior
- **Confirmation Message:** Shows "✅ Respuesta enviada" after successful submission
- **UABC Logo:** Displayed in sidebar header (base64 encoded from `uabc_logo.png`)
- **Team Members:** Listed in Documentation > Team tab
- **Chat (center):** Interactive Q&A with scrollable history, custom avatars 🧑‍💻/🤖
- **Metrics (right panel):** Real-time latency, confidence, SLA, sources
- **Sidebar:** Example Questions + Enhanced Settings (Language, Model, Temperature, Top-K, Response Length, Sources)
- **Feedback:** User rating in right panel with submit button
- **Monitoring:** Performance metrics with Module 17 checklist

**Important:** Upload `streamlit_app_v9.py`, `requirements_v9.txt`, and `uabc_logo.png` to `/content/` before running.

In [11]:
# Instalar dependencias para Streamlit (v9)
!pip -q install -r requirements_v9.txt 2>&1 | grep -v "dependency conflicts" | grep -v "which is incompatible" | grep -v "requires " | grep -v "google-colab" | grep -v "opentelemetry" | grep -v "ERROR: pip"
!pip -q install langchain langchain-openai langchain-community chromadb pypdf 2>&1 | grep -v "dependency conflicts" | grep -v "which is incompatible" | grep -v "requires " | grep -v "google-colab" | grep -v "opentelemetry" | grep -v "ERROR: pip"
print("✅ Dependencias de Streamlit v9 instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.7 MB/s eta 0:00:00
✅ Dependencias de Streamlit v9 instaladas


In [12]:
# Verify dependencies are loaded
import langchain, chromadb, pypdf, streamlit
print(f"langchain:  {langchain.__version__}")
print(f"chromadb:   {chromadb.__version__}")
print(f"pypdf:      {pypdf.__version__}")
print(f"streamlit:  {streamlit.__version__}")
print("\nAll dependencies OK")

langchain:  1.2.10
chromadb:   1.5.0
pypdf:      6.7.1
streamlit:  1.54.0

All dependencies OK


In [13]:
# Install tunnel tools for public URL access
!npm -q install -g localtunnel
!pip -q install pyngrok
print("✅ Tunnel tools installed (localtunnel + ngrok)")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 6s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧✅ Tunnel tools installed (localtunnel + ngrok)


In [14]:
# Kill any previous Streamlit process and launch v9
!pkill -f streamlit 2>/dev/null; sleep 1
!streamlit run /content/streamlit_app_v9.py --server.port 8501 --server.address 0.0.0.0 &>/content/streamlit.log &

^C


In [15]:
# Get your public IP (needed for localtunnel password)
import time
time.sleep(3)  # Wait for Streamlit to start
!curl -s ifconfig.me

34.31.235.66

In [26]:
# ============================================================
# OPTION A (Recommended): ngrok - more reliable than localtunnel
# ============================================================
# 1. Get a free ngrok token at https://dashboard.ngrok.com/signup
# 2. Uncomment the 3 lines below and paste your token:

from pyngrok import ngrok
# ngrok.set_auth_token("PASTE_YOUR_NGROK_TOKEN_HERE")
ngrok.set_auth_token("39pWehIUw8JPAaPS4zE9VuZeoaq_7ZWCJsHfC6CrX9EAVEigd")
ngrok.connect(8501)

# ============================================================
# OPTION B: localtunnel (may fail with "connection refused")
# ============================================================
# If ngrok is not set up, try localtunnel:
#!lt --port 8501


<NgrokTunnel: "https://unpreposterous-tanner-nondeclamatory.ngrok-free.dev" -> "http://localhost:8501">

---
## Module 17: What We Learned

### Key Insights

1. **RAG grounds the LLM effectively** - By restricting responses to retrieved document chunks,
   the system avoids hallucination on most queries. 100% of evaluated responses included
   verifiable citations.

2. **Prompt engineering matters more than model size** - Adding explicit fallback instructions
   ("if not in sources, say so") significantly improved off-topic question handling without
   changing the model.

3. **Chunk size and overlap affect quality** - Increasing `chunk_overlap` from 80 to 150
   improved context coherence in retrieved passages.

4. **Simple metrics reveal a lot** - Tracking just latency, citation presence, and source
   count was enough to identify the main issues and verify fixes.

5. **Error handling is essential for demos** - Adding try/except blocks and loading indicators
   transformed the app from fragile to demo-ready.

6. **UI layout matters for usability** (v3) - Moving metrics to a dedicated right panel
   instead of tabs below the chat eliminated the need for scrolling and made real-time
   monitoring visible at all times alongside the conversation.

### Limitations Observed

- The system only knows what is in the 3 uploaded PDFs (41 pages total, ~97 chunks)
- Latency of ~3-4 seconds per query depends on OpenAI API response time
- No conversation memory across questions (each query is independent)
- ChromaDB is ephemeral in Colab (rebuilds each session)

### Future Improvements (for Module 18+)

- Add conversation memory using `ConversationBufferMemory`
- Implement actual SHAP/LIME explainability analysis
- Deploy permanently to Streamlit Cloud or Hugging Face Spaces
- Add more documents to the knowledge base
- Implement user authentication for production use

---
## Test Questions for the Streamlit UI

Copy-paste these into the chat interface to test:

1. `Que riesgos eticos y recomendaciones institucionales se mencionan sobre el uso de IA?`
2. `Que recomendaciones se dan para integrar la IA generativa en la practica docente?`
3. `Que orientaciones iniciales se proponen para el uso academico de la IA en la universidad?`
4. `Que papel juegan los comites de etica en proyectos de investigacion que usan IA?`
5. `Cual es la capital de Francia?` (should be rejected as off-topic)